In [ ]:
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset
from skimage.filters import gaussian
from sklearn.cluster import KMeans # Moved KMeans import here
import random

def random_geometric_transform(img, heatmaps,
                               p_flip=0.5,
                               max_rot=15,
                               max_trans=20,
                               scale_range=(0.9, 1.1)):

    H, W = heatmaps.shape[1:]

    # ---- Flip horizontal ----
    if random.random() < p_flip:
        img = cv2.flip(img, 1)
        heatmaps = heatmaps[:, :, ::-1]

    # ---- Flip vertical ----
    if random.random() < p_flip:
        img = cv2.flip(img, 0)
        heatmaps = heatmaps[:, ::-1, :]

    # ---- Rotação + escala ----
    angle = random.uniform(-max_rot, max_rot)
    scale = random.uniform(*scale_range)
    center = (W // 2, H // 2)

    M = cv2.getRotationMatrix2D(center, angle, scale)

    img = cv2.warpAffine(img, M, (W, H),
                         flags=cv2.INTER_LINEAR,
                         borderMode=cv2.BORDER_REFLECT)

    for i in range(heatmaps.shape[0]):
        heatmaps[i] = cv2.warpAffine(
            heatmaps[i], M, (W, H),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_REFLECT
        )

    # ---- Translação ----
    tx = random.uniform(-max_trans, max_trans)
    ty = random.uniform(-max_trans, max_trans)
    T = np.float32([[1, 0, tx], [0, 1, ty]])

    img = cv2.warpAffine(img, T, (W, H),
                         borderMode=cv2.BORDER_REFLECT)

    for i in range(heatmaps.shape[0]):
        heatmaps[i] = cv2.warpAffine(
            heatmaps[i], T, (W, H),
            borderMode=cv2.BORDER_REFLECT
        )

    return img, heatmaps


def clean_mask(mask):
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return mask

class WingLandmarkDataset(Dataset):
    # def __init__(self, img_dir, label_dir, num_landmarks,
    #              img_size=(608, 800), sigma=5,
    #              augment=False):

    #     self.img_dir = img_dir
    #     self.label_dir = label_dir
    #     self.files = file_list
    #     self.img_size = img_size
    #     self.sigma = sigma
    #     self.num_landmarks = num_landmarks
    #     self.augment = augment
    #     self.files = []

    def __init__(self, img_dir, label_dir, file_list,
                 num_landmarks,
                 img_size=(608, 800),
                 sigma=5,
                 augment=False):

        self.img_dir = img_dir
        self.label_dir = label_dir
        self.files = file_list
        self.num_landmarks = num_landmarks
        self.img_size = img_size
        self.sigma = sigma
        self.augment = augment

        all_img_files = sorted(os.listdir(img_dir))
        for img_name in all_img_files:
            img_path = os.path.join(img_dir, img_name)
            label_path = os.path.join(label_dir, img_name)

            # Check if both image and label files exist and can be read
            if os.path.exists(img_path) and os.path.exists(label_path):
                temp_img = cv2.imread(img_path)
                temp_label = cv2.imread(label_path)
                if temp_img is not None and temp_label is not None:
                    # Additional check: ensure label has yellow points for clustering
                    # This is important as cluster_landmarks might fail on empty points
                    try:
                        # Resize temp_label for consistent point extraction logic
                        resized_temp_label = cv2.resize(temp_label, self.img_size[::-1])
                        points = self.extract_yellow_points(resized_temp_label)
                        # Check if there are enough points to form clusters
                        if len(points) >= self.num_landmarks:
                           self.files.append(img_name)
                        else:
                           print(f"Skipping {img_name}: Not enough yellow points ({len(points)}) for {self.num_landmarks} landmarks after loading and resizing.")
                    except Exception as e:
                        print(f"Skipping {img_name}: Error processing yellow points or clustering - {e}")
                else:
                    print(f"Skipping invalid or unreadable file pair (imread returned None): {img_name}")
            else:
                print(f"Skipping missing file pair: {img_name} (img exists: {os.path.exists(img_path)}, label exists: {os.path.exists(label_path)}).")

        if not self.files:
            raise RuntimeError("No valid image files found in the specified directories for training after filtering.")

    def extract_yellow_points(self, label_img):
        hsv = cv2.cvtColor(label_img, cv2.COLOR_BGR2HSV)
        # lower_yellow = np.array([25, 100, 100])
        # upper_yellow = np.array([35, 255, 255])
        lower_yellow = np.array([15, 200, 200])
        upper_yellow = np.array([35, 255, 255])

        mask1 = cv2.inRange(hsv, lower_yellow, upper_yellow)
        points1 = np.column_stack(np.where(mask1 > 0)) # points are (y, x) format here
        # plt.figure(figsize=(6,6))
        # plt.imshow(cv2.cvtColor(mask1, cv2.COLOR_BGR2RGB))
        # plt.axis("off")
        # plt.show()

        # print(f"Extracted {len(points1)} yellow points from {label_img.shape} label image.")
        mask = clean_mask(mask1)
        points = np.column_stack(np.where(mask > 0)) # points are (y, x) format here
        # print(f"Cleaned - Extracted {len(points)} yellow points from {label_img.shape} label image.")
        # plt.figure(figsize=(6,6))
        # plt.imshow(cv2.cvtColor(mask, cv2.COLOR_BGR2RGB))
        # plt.axis("off")
        # plt.show()

        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask)
        centers = []
        for i in range(1, num_labels):  # ignora fundo
            # print(f"Processing label {i} out of {num_labels - 1} labels.")
            area = stats[i, cv2.CC_STAT_AREA]
            # print(area)
            if 1 < area < 1000:
                c = centroids[i]
                centers.append([int(c[1]), int(c[0])])  # (y, x)

        vis = label_img.copy()
        for y, x in centers:
            cv2.circle(vis, (x, y), 5, (0, 0, 255), -1)

        # plt.figure(figsize=(6,6))
        # plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
        # plt.title(f"{len(centers)} landmarks detectados")
        # plt.axis("off")
        # plt.show()

        return points

    def cluster_landmarks(self, points):
        """Assume pontos bem separados → divide por K-means"""
        if len(points) < self.num_landmarks:
             # This case should ideally be caught in __init__
             # If it still occurs, it's a data issue that was missed or a dynamic change
             raise ValueError(f"Insufficient points ({len(points)}) for KMeans clustering with {self.num_landmarks} clusters.")

        kmeans = KMeans(n_clusters=self.num_landmarks, n_init=10, random_state=42) # Added random_state for reproducibility
        centers = kmeans.fit(points).cluster_centers_
        return centers.astype(int)

    def generate_heatmaps(self, centers):
        heatmaps = np.zeros((self.num_landmarks,
                              self.img_size[0],
                              self.img_size[1]), dtype=np.float32)
        for i, (y, x) in enumerate(centers):
            # Ensure coordinates are within bounds before setting to 1.0
            x = int(np.clip(x, 0, self.img_size[1] - 1))
            y = int(np.clip(y, 0, self.img_size[0] - 1))
            heatmaps[i, y, x] = 1.0
            heatmaps[i] = gaussian(heatmaps[i], sigma=self.sigma)
            # Handle case where heatmap might be all zeros (e.g., if landmark was too close to border or filtered entirely)
            max_val = heatmaps[i].max()
            if max_val > 0:
                heatmaps[i] /= max_val
            else:
                heatmaps[i] = 0.0 # Keep it 0 if no landmark contributes or it was out of bounds
        return heatmaps

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        name = self.files[idx]
        img_path = os.path.join(self.img_dir, name)
        label_path = os.path.join(self.label_dir, name)

        img = cv2.imread(img_path)
        label = cv2.imread(label_path)

        # These checks should ideally pass due to __init__ filtering
        if img is None:
            raise RuntimeError(f"Failed to load image at {img_path} even after __init__ check. File might have been corrupted or moved.")
        if label is None:
            raise RuntimeError(f"Failed to load label image at {label_path} even after __init__ check. File might have been corrupted or moved.")

        # Perform resize
        img = cv2.resize(img, self.img_size[::-1])
        label = cv2.resize(label, self.img_size[::-1])

########## inicio DA
        points = self.extract_yellow_points(label)
        if len(points) < self.num_landmarks:
          raise ValueError("Insufficient landmarks")

        centers = self.cluster_landmarks(points)
        heatmaps = self.generate_heatmaps(centers)

        if self.augment:
            img, heatmaps = random_geometric_transform(img, heatmaps)
########## fim DA

        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2, 0, 1))

        points = self.extract_yellow_points(label)
        # Re-check for sufficient points, in case filtering in __init__ wasn't perfect or data changed dynamically
        if len(points) < self.num_landmarks:
            # If this happens, it implies a data inconsistency
            raise ValueError(f"Not enough yellow points in label for {name} ({len(points)}) for {self.num_landmarks} landmarks during __getitem__ after filtering.")

        centers = self.cluster_landmarks(points)
        heatmaps = self.generate_heatmaps(centers)

        return (
            torch.tensor(img, dtype=torch.float32),
            torch.tensor(heatmaps, dtype=torch.float32)
        )

In [ ]:
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, out_channels):
        super().__init__()

        self.enc1 = DoubleConv(3, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)

        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(512, 1024)

        self.up4 = nn.ConvTranspose2d(1024, 512, 2, 2)
        self.dec4 = DoubleConv(1024, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec3 = DoubleConv(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec2 = DoubleConv(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec1 = DoubleConv(128, 64)

        self.out = nn.Conv2d(64, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.dec4(torch.cat([self.up4(b), e4], 1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))

        return torch.sigmoid(self.out(d1))


In [ ]:
def soft_argmax_2d(heatmaps, beta=100):
    B, C, H, W = heatmaps.shape
    heatmaps = heatmaps.view(B, C, -1)
    softmax = torch.softmax(heatmaps * beta, dim=-1)

    xs = torch.linspace(0, W - 1, W, device=heatmaps.device)
    ys = torch.linspace(0, H - 1, H, device=heatmaps.device)
    ys, xs = torch.meshgrid(ys, xs, indexing="ij")

    xs = xs.reshape(-1)
    ys = ys.reshape(-1)

    x = torch.sum(xs * softmax, dim=-1)
    y = torch.sum(ys * softmax, dim=-1)

    return torch.stack([x, y], dim=-1)


In [ ]:
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import os
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


device = "cuda" if torch.cuda.is_available() else "cpu"
NUM_LANDMARKS = 30

# all_files = sorted(os.listdir("/content/drive/MyDrive/Colab Notebooks/ABELHAS/Plega/Base2025/Asas Anteriores Com Landmarks"))
VALID_EXTENSIONS = (".jpg", ".jpeg", ".png")
def list_image_files(folder):
    return sorted([
        f for f in os.listdir(folder)
        if f.lower().endswith(VALID_EXTENSIONS)
    ])

all_files = list_image_files("/content/drive/MyDrive/Colab Notebooks/ABELHAS/Plega/Base2025/Asas Anteriores Com Landmarks")

train_files, temp_files = train_test_split(
    all_files, test_size=0.3, random_state=42
)

val_files, test_files = train_test_split(
    temp_files, test_size=0.5, random_state=42
)

train_dataset = WingLandmarkDataset(
    "/content/drive/MyDrive/Colab Notebooks/ABELHAS/Plega/Base2025/Asas Anteriores de Plega hagenella",
    "/content/drive/MyDrive/Colab Notebooks/ABELHAS/Plega/Base2025/Asas Anteriores Com Landmarks",
    train_files,
    NUM_LANDMARKS,
    augment=True
)

val_dataset = WingLandmarkDataset(
    "/content/drive/MyDrive/Colab Notebooks/ABELHAS/Plega/Base2025/Asas Anteriores de Plega hagenella",
    "/content/drive/MyDrive/Colab Notebooks/ABELHAS/Plega/Base2025/Asas Anteriores Com Landmarks",
    val_files,
    NUM_LANDMARKS,
    augment=False
)

test_dataset = WingLandmarkDataset(
    "/content/drive/MyDrive/Colab Notebooks/ABELHAS/Plega/Base2025/Asas Anteriores de Plega hagenella",
    "/content/drive/MyDrive/Colab Notebooks/ABELHAS/Plega/Base2025/Asas Anteriores Com Landmarks",
    test_files,
    NUM_LANDMARKS,
    augment=False
)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=1, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=1, shuffle=False)

# dataset = WingLandmarkDataset(
#     "/content/drive/MyDrive/Colab Notebooks/ABELHAS/Plega/Base2025/Asas Anteriores de Plega hagenella",
#     "/content/drive/MyDrive/Colab Notebooks/ABELHAS/Plega/Base2025/Asas Anteriores Com Landmarks",
#     num_landmarks=NUM_LANDMARKS#,        augment=config["augment"]

# )

# train_size = int(0.8 * len(dataset))
# val_size = len(dataset) - train_size
# train_set, val_set = random_split(dataset, [train_size, val_size])

# train_loader = DataLoader(train_set, batch_size=2, shuffle=True)
# val_loader = DataLoader(val_set, batch_size=1)

model = UNet(out_channels=NUM_LANDMARKS).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

train_losses, val_losses = [], []

epochs = 10
for epoch in range(epochs):
    model.train()
    t_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        loss = criterion(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        t_loss += loss.item()

    model.eval()
    v_loss = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            v_loss += criterion(model(x), y).item()

    train_losses.append(t_loss / len(train_loader))
    val_losses.append(v_loss / len(val_loader))

    print(f"Epoch {epoch+1}: Train={train_losses[-1]:.4f} | Val={val_losses[-1]:.4f}")


Epoch 1: Train=0.2289 | Val=0.2054
Epoch 2: Train=0.1922 | Val=0.1837
Epoch 3: Train=0.1690 | Val=0.1591
Epoch 4: Train=0.1500 | Val=0.1426


In [ ]:
plt.plot(train_losses, label="Treino")
plt.plot(val_losses, label="Validação")
plt.xlabel("Épocas")
plt.ylabel("Loss (MSE)")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
def evaluate(model, dataloader):
    model.eval()
    errors = []

    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            y = y.to(device)

            pred = model(x)

            pred_pts = soft_argmax_2d(pred)
            gt_pts = soft_argmax_2d(y)

            error = torch.norm(pred_pts - gt_pts, dim=-1)
            errors.extend(error.cpu().numpy().flatten())

    errors = np.array(errors)
    return {
        "Mean Error (px)": errors.mean(),
        "Std (px)": errors.std(),
        "RMSE (px)": np.sqrt((errors**2).mean())
    }

metrics = evaluate(model, val_loader)
for k, v in metrics.items():
    print(f"{k}: {v:.2f}")


In [ ]:
def visualize(model, dataset, idx=0):
    model.eval()
    img, gt = dataset[idx]
    with torch.no_grad():
        pred = model(img.unsqueeze(0).to(device)).cpu()[0]

    img = img.permute(1, 2, 0).numpy()

    fig, ax = plt.subplots(1, 3, figsize=(25, 15))
    ax[0].imshow(img)
    ax[0].set_title("Imagem")

    ax[1].imshow(gt.sum(0), cmap="hot")
    ax[1].set_title("GT Heatmaps")

    ax[2].imshow(img)
    ax[2].imshow(pred.sum(0), cmap="jet", alpha=0.5)
    ax[2].set_title("Predição")

    for a in ax:
        a.axis("off")
    plt.show()

visualize(model, dataset, idx=35)